In [ ]:
import os
import requests
import zipfile
from tqdm import tqdm

# -------- CONFIG --------
DATASET_URL = "https://unsplash.com/data/lite/latest" #download link for lite dataset
SAVE_DIR = "data"
ZIP_NAME = "unsplash_lite.zip"

os.makedirs(SAVE_DIR, exist_ok=True)

zip_path = os.path.join(SAVE_DIR, ZIP_NAME)

def download_file(url, save_path):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(save_path, 'wb') as file, tqdm(
        desc="Downloading",
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                file.write(chunk)
                bar.update(len(chunk))

#unzip
def unzip_file(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

#delete zip
def delete_file(path):
    os.remove(path)

#downloads only if zip doesnt exist
if not os.path.exists(zip_path):
    print("Downloading dataset...")
    download_file(DATASET_URL, zip_path)
else:
    print("Zip already exists, skipping download.")

print("Extracting dataset...")
unzip_file(zip_path, SAVE_DIR)

print("Deleting zip file...")
delete_file(zip_path)

print("✅ Dataset ready!")

Downloading: 100%|██████████| 312M/312M [02:29<00:00, 2.19MB/s]   


Extracting dataset...
Deleting zip file...
✅ Dataset ready!


In [11]:
import numpy as np
import pandas as pd
import glob

In [12]:
import os
import glob
import pandas as pd

path = "data"
documents = ['photos', 'keywords', 'collections', 'conversions', 'colors']
datasets = {}

print("Checking base path:", os.path.abspath(path))

if not os.path.exists(path):
    raise FileNotFoundError("Path does not exist")

print("\nDirectory contents:")
for f in os.listdir(path):
    print(" ", f)

print("\nLoading datasets...\n")

for doc in documents:
    print("Processing:", doc)

    patterns = [
        os.path.join(path, f"{doc}.csv*"),
        os.path.join(path, f"{doc}.tsv*"),
        os.path.join(path, f"{doc}*")
    ]

    files = []
    for p in patterns:
        matched = glob.glob(p)
        files.extend(matched)

    files = sorted(list(set(files)))

    print(" Files found:", len(files))
    for f in files:
        print("  ", f)

    subsets = []

    for filename in files:
        print(" Reading:", filename)

        try:
            df = pd.read_csv(filename, sep="\t")
        except Exception as e:
            print("  Failed TSV read:", e)
            continue

        print("  Shape:", df.shape)
        subsets.append(df)

    if len(subsets) > 0:
        datasets[doc] = pd.concat(subsets, ignore_index=True)
        print(" Final shape:", datasets[doc].shape)
    else:
        print(" No valid data loaded")

    print()

print("DONE")

Checking base path: c:\Users\Madhav\Desktop\Madhav\ANU\Semester 1, 2026\COMP3242 Deep Learning\Project\AsciiVision\data

Directory contents:
  collections.csv000
  colors.csv000
  conversions.csv000
  DOCS.md
  keywords.csv000
  photos.csv000
  README.md
  TERMS.md

Loading datasets...

Processing: photos
 Files found: 1
   data\photos.csv000
 Reading: data\photos.csv000
  Shape: (25000, 31)
 Final shape: (25000, 31)

Processing: keywords
 Files found: 1
   data\keywords.csv000
 Reading: data\keywords.csv000
  Shape: (755782, 6)
 Final shape: (755782, 6)

Processing: collections
 Files found: 1
   data\collections.csv000
 Reading: data\collections.csv000
  Shape: (2144835, 5)
 Final shape: (2144835, 5)

Processing: conversions
 Files found: 1
   data\conversions.csv000
 Reading: data\conversions.csv000
  Shape: (6246063, 6)
 Final shape: (6246063, 6)

Processing: colors
 Files found: 1
   data\colors.csv000
 Reading: data\colors.csv000
  Shape: (244953, 8)
 Final shape: (244953, 8)

DO

In [21]:
photos = datasets["photos"]
photos.iloc[456]["photo_image_url"]

'https://images.unsplash.com/photo-1447127480722-84cc5d1c5dfe'

In [22]:
import pandas as pd
import re

keywords = datasets["keywords"]

# -----------------------------
# 1. Basic cleanup
# -----------------------------
kw = keywords.copy()

# normalize text
kw["keyword"] = kw["keyword"].astype(str).str.lower().str.strip()

# remove weird characters / keep only letters, spaces, hyphens
kw["keyword_clean"] = kw["keyword"].apply(
    lambda x: re.sub(r"[^a-z\s\-]", "", x)
)

# remove empty strings
kw = kw[kw["keyword_clean"].str.len() > 0]

# -----------------------------
# 2. Most common keywords
# -----------------------------
top_keywords = kw["keyword_clean"].value_counts()

print("Top 20 keywords:\n")
print(top_keywords.head(20))

# -----------------------------
# 3. Simple keyword grouping (themes)
# -----------------------------
themes = {
    "nature": ["mountain", "forest", "tree", "river", "lake", "ocean", "sea", "sky", "cloud", "sunset", "sunrise", "landscape"],
    "people": ["man", "woman", "people", "portrait", "face", "girl", "boy"],
    "city": ["city", "street", "building", "architecture", "urban", "skyscraper"],
    "animals": ["dog", "cat", "bird", "horse", "wildlife", "lion", "bear"],
    "objects": ["car", "phone", "computer", "table", "chair", "book"],
    "abstract": ["background", "texture", "pattern", "wallpaper"]
}

def assign_theme(word):
    for theme, words in themes.items():
        if word in words:
            return theme
    return "other"

kw["theme"] = kw["keyword_clean"].apply(assign_theme)

theme_counts = kw["theme"].value_counts()

print("\nKeyword themes:\n")
print(theme_counts)

# -----------------------------
# 4. Top keywords per theme
# -----------------------------
print("\nTop keywords per theme:\n")

for t in theme_counts.index:
    subset = kw[kw["theme"] == t]["keyword_clean"].value_counts().head(10)
    print(f"\n[{t}]")
    print(subset)

Top 20 keywords:

keyword_clean
nature        18661
outdoors      16590
plant         12192
landscape      8616
animal         8471
scenery        8326
tree           8120
water          7909
sky            7244
mountain       6713
sea            5807
land           5783
ocean          5468
building       4918
vegetation     4613
light          4292
snow           4156
person         4116
flower         4112
blossom        3959
Name: count, dtype: int64

Keyword themes:

theme
other       666302
nature       58074
city         11369
animals       8894
abstract      5619
people        4097
objects       1217
Name: count, dtype: int64

Top keywords per theme:


[other]
keyword_clean
nature        18661
outdoors      16590
plant         12192
animal         8471
scenery        8326
water          7909
land           5783
vegetation     4613
light          4292
snow           4156
Name: count, dtype: int64

[nature]
keyword_clean
landscape    8616
tree         8120
sky          7244
mounta

In [52]:
import random
# -----------------------------
# Load datasets
# -----------------------------
keywords_df = datasets["keywords"].copy()
photos_df = datasets["photos"].copy()

# -----------------------------
# Normalize keywords
# -----------------------------
keywords_df["keyword"] = keywords_df["keyword"].str.lower().str.strip()

# -----------------------------
# Step 1: define keywords
# -----------------------------
MOUNTAIN_KEYWORDS = [
    "mountain", "mountains", "hill", "valley"
]

BAD_KEYWORDS = [
    "building", "road", "city", "urban",
    "flare", "red sky",
    "adventure", "leisure activities",
    "shoreline", "beach", "coast"
]

# -----------------------------
# Step 2: get mountain keywords
# -----------------------------
kw_filtered = keywords_df[
    keywords_df["keyword"].isin(MOUNTAIN_KEYWORDS)
]

print("Initial mountain keyword rows:", len(kw_filtered))
print("Initial unique photo IDs:", kw_filtered["photo_id"].nunique())

# -----------------------------
# Step 3: find bad photo IDs
# -----------------------------
bad_photo_ids = keywords_df[
    keywords_df["keyword"].isin(BAD_KEYWORDS)
]["photo_id"].unique()

print("Bad photo IDs:", len(bad_photo_ids))

# -----------------------------
# Step 4: remove bad keyword contamination
# -----------------------------
clean_kw = kw_filtered[
    ~kw_filtered["photo_id"].isin(bad_photo_ids)
]

print("Clean keyword rows:", len(clean_kw))
print("Clean unique photo IDs:", clean_kw["photo_id"].nunique())

photo_ids = clean_kw["photo_id"].unique()

# -----------------------------
# Step 5: join with photos
# -----------------------------
mountain_photos = photos_df[
    photos_df["photo_id"].isin(photo_ids)
].copy()

print("After join:", len(mountain_photos))

# -----------------------------
# Step 6: landscape filter
# -----------------------------
mountain_landscape_photos = mountain_photos[
    mountain_photos["photo_width"] > mountain_photos["photo_height"]
].copy()

print("Landscape photos:", len(mountain_landscape_photos))

# -----------------------------
# Step 7: select useful columns
# -----------------------------
mountain_landscape_photos = mountain_landscape_photos[
    [
        "photo_id",
        "photo_width",
        "photo_height",
        "photo_image_url",
        "photo_description",
        "photographer_username"
    ]
]

# -----------------------------
# Step 8: reset index
# -----------------------------
mountain_landscape_photos = mountain_landscape_photos.reset_index(drop=True)

print("Final dataset size:", len(mountain_landscape_photos))

# -----------------------------
# Step 9: sample some URLs
# -----------------------------
print("\nSample image URLs:\n")

for _ in range(20):
    idx = random.randint(0, len(mountain_landscape_photos) - 1)
    print(mountain_landscape_photos.loc[idx, "photo_image_url"])


Initial mountain keyword rows: 10476
Initial unique photo IDs: 7027
Bad photo IDs: 12440
Clean keyword rows: 3096
Clean unique photo IDs: 2025
After join: 2025
Landscape photos: 1368
Final dataset size: 1368

Sample image URLs:

https://images.unsplash.com/photo-1443904449649-1510ad809291
https://images.unsplash.com/photo-1463101140509-cb3dad26e35d
https://images.unsplash.com/photo-1562004736-6704d0518d24
https://images.unsplash.com/photo-1573537677396-35af5d5291fc
https://images.unsplash.com/photo-1578164787304-b27ce2431417
https://images.unsplash.com/photo-1583507264623-e424bcc92e90
https://images.unsplash.com/photo-1526402978125-f1d6df91cbac
https://images.unsplash.com/photo-1542750646-9418b4cc8808
https://images.unsplash.com/photo-1424886097867-7a53e6058dff
https://images.unsplash.com/photo-1462480803487-a2edfd796460
https://images.unsplash.com/photo-1492553578468-c989392a45be
https://images.unsplash.com/photo-1579621982539-3c5456b46900
https://images.unsplash.com/photo-14463201082

In [51]:
# -----------------------------
# Step 1: define mountain keywords
# -----------------------------
MOUNTAIN_KEYWORDS = [
    "mountain", "mountains", "hill", "valley"
]

# normalize keywords
keywords_df["keyword"] = keywords_df["keyword"].str.lower().str.strip()

# -----------------------------
# Step 2: get mountain photo_ids
# -----------------------------
mountain_ids = keywords_df[
    keywords_df["keyword"].isin(MOUNTAIN_KEYWORDS)
]["photo_id"].unique()

print("Mountain photo count:", len(mountain_ids))

# -----------------------------
# Step 3: get ALL keywords for those photos
# -----------------------------
mountain_all_keywords = keywords_df[
    keywords_df["photo_id"].isin(mountain_ids)
].copy()

print("Total keyword rows (expanded):", len(mountain_all_keywords))

# -----------------------------
# Step 4: count co-occurring keywords
# -----------------------------
keyword_counts = (
    mountain_all_keywords["keyword"]
    .value_counts()
)

# -----------------------------
# Step 5: remove the base keywords themselves
# -----------------------------
keyword_counts = keyword_counts.drop(
    labels=MOUNTAIN_KEYWORDS,
    errors="ignore"
)

# -----------------------------
# Step 6: show results
# -----------------------------
print("\nTop co-occurring keywords:\n")
print(keyword_counts.head(50))

Mountain photo count: 7027
Total keyword rows (expanded): 280173

Top co-occurring keywords:

keyword
nature                   7366
outdoors                 6936
landscape                5050
scenery                  4901
mountain range           3932
sky                      3553
plant                    3253
snow                     3028
tree                     2998
ice                      2910
water                    2865
land                     2700
wilderness               2408
plateau                  2289
peak                     2151
glacier                  2121
sea                      2021
weather                  1970
building                 1955
ocean                    1945
countryside              1806
cliff                    1747
fir                      1703
cloud                    1680
vegetation               1652
abies                    1633
road                     1623
adventure                1612
rock                     1588
sunset                   158

In [53]:
import os
import cv2
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# -----------------------------
# CONFIG
# -----------------------------
SAVE_DIR = "cleaned"
CSV_PATH = os.path.join(SAVE_DIR, "metadata.csv")

TARGET_WIDTH = 256
TARGET_HEIGHT = 144

MAX_WORKERS = 32   # adjust based on your internet (8–32 is fine)

os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------
# Helper: download image
# -----------------------------
def download_image(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.content
    except:
        return None
    return None

# -----------------------------
# Helper: crop to 16:9 (no distortion)
# -----------------------------
def crop_to_aspect(img, target_ratio=16/9):
    h, w = img.shape[:2]
    current_ratio = w / h

    if current_ratio > target_ratio:
        # too wide → crop width
        new_w = int(h * target_ratio)
        start = (w - new_w) // 2
        img = img[:, start:start+new_w]
    else:
        # too tall → crop height
        new_h = int(w / target_ratio)
        start = (h - new_h) // 2
        img = img[start:start+new_h, :]

    return img

# -----------------------------
# Helper: process one image
# -----------------------------
def process_row(row):
    photo_id = row["photo_id"]
    url = row["photo_image_url"]

    img_bytes = download_image(url)
    if img_bytes is None:
        return None

    # decode
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if img is None:
        return None

    try:
        # crop to 16:9
        img = crop_to_aspect(img)

        # resize
        img = cv2.resize(img, (TARGET_WIDTH, TARGET_HEIGHT))

        # save
        filename = f"{photo_id}.jpg"
        save_path = os.path.join(SAVE_DIR, filename)

        cv2.imwrite(save_path, img, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

        return {
            "photo_id": photo_id,
            "filename": filename,
            "width": TARGET_WIDTH,
            "height": TARGET_HEIGHT,
            "original_url": url,
            "description": row.get("photo_description", ""),
            "author": row.get("photographer_username", "")
        }

    except:
        return None

# -----------------------------
# MAIN PIPELINE
# -----------------------------
import numpy as np

results = []

rows = mountain_landscape_photos.to_dict("records")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, row) for row in rows]

    for future in tqdm(as_completed(futures), total=len(futures)):
        res = future.result()
        if res is not None:
            results.append(res)

# -----------------------------
# Save CSV
# -----------------------------
df = pd.DataFrame(results)
df.to_csv(CSV_PATH, index=False)

print("Done.")
print("Images saved:", len(df))
print("CSV saved at:", CSV_PATH)

100%|██████████| 1368/1368 [02:22<00:00,  9.57it/s]

Done.
Images saved: 1368
CSV saved at: cleaned\metadata.csv


In [54]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

# -----------------------------
# CONFIG
# -----------------------------
INPUT_DIR = "dataset/faces"
OUTPUT_DIR = "dataset/faces_compressed"
CSV_PATH = os.path.join(OUTPUT_DIR, "data.csv")

TARGET_SIZE = 128  # 128x128

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Process images
# -----------------------------
data = []

files = sorted(os.listdir(INPUT_DIR))

for filename in tqdm(files):
    input_path = os.path.join(INPUT_DIR, filename)

    # skip non-images just in case
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    # read image
    img = cv2.imread(input_path)
    if img is None:
        continue

    try:
        # resize (no crop needed, already square)
        img_resized = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE))

        # output filename (same name)
        output_path = os.path.join(OUTPUT_DIR, filename)

        # save
        cv2.imwrite(output_path, img_resized, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

        # store metadata
        data.append({
            "id": os.path.splitext(filename)[0],   # "1", "2", ...
            "filename": filename,
            "path": output_path,
            "width": TARGET_SIZE,
            "height": TARGET_SIZE
        })

    except:
        continue

# -----------------------------
# Save CSV
# -----------------------------
df = pd.DataFrame(data)
df.to_csv(CSV_PATH, index=False)

print("Done.")
print("Images processed:", len(df))
print("CSV saved at:", CSV_PATH)

100%|██████████| 9999/9999 [03:40<00:00, 45.43it/s]


Done.
Images processed: 9999
CSV saved at: dataset/faces_compressed\data.csv
